In [ ]:
import zipfile
import os

zip_path = '/content/has_dat.zip'   # change this path
extract_dir = '/content/dataset_hassan'                       # where to extract

# Create directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Extract directly from the file path
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("✅ Dataset extracted to:", extract_dir)


In [ ]:
!pip install -q lz4
!pip install -q keras-facenet
!pip install -q mtcnn
!pip install -q scikit-learn

In [ ]:
pip install lz4

In [ ]:
import numpy as np
import os
import cv2
import pickle
from mtcnn import MTCNN
from keras_facenet import FaceNet
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import joblib

In [ ]:
facenet = FaceNet()
detector = MTCNN()


In [ ]:
def load_faces_from_directory(directory):
    """
    Load images and extract face embeddings using FaceNet
    """
    X = []
    y = []

    classes = sorted(os.listdir(directory))
    print(f"\n📁 Found classes: {classes}")

    for label in classes:
        class_path = os.path.join(directory, label)
        if not os.path.isdir(class_path):
            continue

        print(f"\n🔄 Processing class '{label}'...")
        images = os.listdir(class_path)

        for img_name in images:
            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)
            if img is None:
                print(f"  ⚠️ Could not read {img_name}")
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            detections = detector.detect_faces(img_rgb)

            if len(detections) == 0:
                print(f" No face detected in {img_name}")
                continue

            detection = max(detections, key=lambda x: x['confidence'])
            x1, y1, w, h = detection['box']
            x2, y2 = x1 + w, y1 + h

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_rgb.shape[1], x2), min(img_rgb.shape[0], y2)

            face = img_rgb[y1:y2, x1:x2]

            face = cv2.resize(face, (160, 160))


            face_expanded = np.expand_dims(face, axis=0)
            embedding = facenet.embeddings(face_expanded)[0]

            X.append(embedding)
            y.append(label)
            print(f"  Processed {img_name}")

    return np.array(X), np.array(y)


In [ ]:
train_dir = '/content/dataset_hassan/DAta sets/train'
test_dir = '/content/dataset_hassan/DAta sets/test'

print("\n" + "="*50)
print("LOADING TRAINING DATA")
print("="*50)

X_train, y_train = load_faces_from_directory(train_dir)

print("\n" + "="*50)
print("LOADING TESTING DATA")
print("="*50)

X_test, y_test = load_faces_from_directory(test_dir)

print("\n" + "="*50)
print("DATASET SUMMARY")
print("="*50)

print(f"\nTraining Data:")
print(f"  Total samples: {len(X_train)}")
print(f"  Embedding shape: {X_train[0].shape}")
print(f"  Classes: {np.unique(y_train)}")
print(f"  Samples per class: {dict(zip(*np.unique(y_train, return_counts=True)))}")

print(f"\nTesting Data:")
print(f"  Total samples: {len(X_test)}")
print(f"  Classes: {np.unique(y_test)}")
print(f"  Samples per class: {dict(zip(*np.unique(y_test, return_counts=True)))}")

In [ ]:
print("Unique classes in training data:", np.unique(y_train))
print("Unique classes in testing data:", np.unique(y_test))

In [ ]:
encoder = LabelEncoder()
encoder.fit(np.concatenate([y_train, y_test]))  # Fit on all labels
y_train_encoded = encoder.transform(y_train)
y_test_encoded = encoder.transform(y_test)


In [ ]:
print("\n" + "="*50)
print("TRAINING SVM CLASSIFIER")
print("="*50)
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train, y_train_encoded)


In [ ]:
print("\n" + "="*50)
print("EVALUATION")
print("="*50)
y_train_pred = svm_model.predict(X_train)
train_acc = accuracy_score(y_train_encoded, y_train_pred)
print(f"\n📈 Training Accuracy: {train_acc*100:.2f}%")
y_test_pred = svm_model.predict(X_test)
test_acc = accuracy_score(y_test_encoded, y_test_pred)
print(f"📈 Testing Accuracy: {test_acc*100:.2f}%")
print("\n📋 Classification Report (Test Set):")
print(classification_report(y_test_encoded, y_test_pred, target_names=encoder.classes_))

In [ ]:
print("\n" + "="*50)
print("EVALUATION")
print("="*50)
y_train_pred = svm_model.predict(X_train)
train_acc = accuracy_score(y_train_encoded, y_train_pred)
print(f"\n📈 Training Accuracy: {train_acc*100:.2f}%")
y_test_pred = svm_model.predict(X_test)
test_acc = accuracy_score(y_test_encoded, y_test_pred)
print(f"📈 Testing Accuracy: {test_acc*100:.2f}%")
print("\n📋 Classification Report (Test Set):")
print(classification_report(y_test_encoded, y_test_pred, target_names=encoder.classes_))

In [ ]:
print("Unique classes in training data:", np.unique(y_train))
print("Unique classes in testing data:", np.unique(y_test))

In [ ]:
print("\n" + "="*50)
print("EVALUATION")
print("="*50)
y_train_pred = svm_model.predict(X_train)
train_acc = accuracy_score(y_train_encoded, y_train_pred)
print(f"\n📈 Training Accuracy: {train_acc*100:.2f}%")
y_test_pred = svm_model.predict(X_test)
test_acc = accuracy_score(y_test_encoded, y_test_pred)
print(f"📈 Testing Accuracy: {test_acc*100:.2f}%")
print("\n📋 Classification Report (Test Set):")
print(classification_report(y_test_encoded, y_test_pred, target_names=encoder.classes_))

In [ ]:
def predict_face(image_path):
    """
    Predict the identity of a face in an image
    """
    img = cv2.imread(image_path)
    if img is None:
        return "Error: Could not read image", 0.0

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    detections = detector.detect_faces(img_rgb)

    if len(detections) == 0:
        return "No face detected", 0.0

    detection = max(detections, key=lambda x: x['confidence'])
    x1, y1, w, h = detection['box']
    x2, y2 = x1 + w, y1 + h

    face = img_rgb[max(0,y1):min(img_rgb.shape[0],y2),
                   max(0,x1):min(img_rgb.shape[1],x2)]
    face = cv2.resize(face, (160, 160))

    face_expanded = np.expand_dims(face, axis=0)
    embedding = facenet.embeddings(face_expanded)


    prediction = svm_model.predict(embedding)
    probability = svm_model.predict_proba(embedding)

    predicted_label = encoder.inverse_transform(prediction)[0]
    confidence = np.max(probability) * 100

    return predicted_label, confidence

In [ ]:
label, confidence = predict_face('/d5328cb7-64ac-4fe8-9dbb-99def9aa5a5a.jfif')
print(f'Predicted: {label} (Confidence: {confidence:.2f}%)')

In [ ]:
joblib.dump(svm_model, "face_recognition_svm.pkl")
joblib.dump(encoder, "label_encoder.pkl")
print("✅ Models saved: face_recognition_svm.pkl & label_encoder.pkl")

In [ ]:
import pickle

model_data = {
    'model': svm_model,
    'encoder': encoder
}

with open('face_model123.pkl', 'wb') as f:
    pickle.dump(model_data, f)
